In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
train_path = "/content/drive/MyDrive/MajorProject/CIC_IoT_2023/Full_Dataset/augmented_train_data.csv"
test_path  = "/content/drive/MyDrive/MajorProject/CIC_IoT_2023/Full_Dataset/test_frozen.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (912851, 47)
Test shape: (259646, 47)


/tmp/ipykernel_12348/1898976211.py:5: DtypeWarning: Columns (42) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df  = pd.read_csv(test_path)


### Data cleaning

In [5]:
print(f"Shape of training data: {train_df.shape}")
print(f"Shape of test data: {test_df.shape}")

Shape of training data: (912851, 47)
Shape of test data: (259646, 47)


In [6]:
LABEL_COL = 'label'

train_df = train_df.dropna(subset=[LABEL_COL])
test_df = test_df.dropna(subset=[LABEL_COL])

print("Train shape after cleaning", train_df.shape)
print("Test shape after cleaning", test_df.shape)

Train shape after cleaning (912851, 47)
Test shape after cleaning (259645, 47)


In [7]:
test_df["Radius"] = pd.to_numeric(test_df["Radius"], errors="coerce")

### Split features and target

In [8]:
X_train = train_df.drop(columns=[LABEL_COL])
y_train = train_df[LABEL_COL]

X_test = test_df.drop(columns=[LABEL_COL])
y_test = test_df[LABEL_COL]

### Label encoding

In [9]:
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print("Number of classes:", len(le.classes_))

Number of classes: 34


### Convert to NumPy arrays for Random Forest

In [10]:
# Random Forest in sklearn works directly on tabular arrays
X_train_rf = X_train.values
X_test_rf = X_test.values

### Define Random Forest

In [11]:
rf_params = {
    "n_estimators": 260,
    "criterion": "gini",
    "max_depth": 24,
    "min_samples_split": 6,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
    "max_samples": 0.7,
    "class_weight": "balanced_subsample",
    "n_jobs": -1,
    "random_state": 42
}

In [12]:
model = RandomForestClassifier(**rf_params)
model.fit(X_train_rf, y_train_enc)

RandomForestClassifier(class_weight='balanced_subsample', max_depth=24,
                       max_samples=0.7, min_samples_leaf=2, min_samples_split=6,
                       n_estimators=260, n_jobs=-1, random_state=42)

In [13]:
y_pred_probs = model.predict_proba(X_test_rf)
y_pred = model.predict(X_test_rf)

In [14]:
print("Accuracy:", accuracy_score(y_test_enc, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test_enc, y_pred, target_names=le.classes_))

Accuracy: 0.8999017889811088

Classification Report:

                         precision    recall  f1-score   support

       Backdoor_Malware       0.59      0.56      0.57       638
          BenignTraffic       0.55      0.84      0.67      6000
       BrowserHijacking       0.61      0.56      0.58      1167
       CommandInjection       0.26      0.78      0.39      1078
 DDoS-ACK_Fragmentation       1.00      0.99      1.00      7799
        DDoS-HTTP_Flood       1.00      0.99      0.99      5742
        DDoS-ICMP_Flood       1.00      1.00      1.00      6000
DDoS-ICMP_Fragmentation       1.00      0.99      1.00      7655
      DDoS-PSHACK_Flood       1.00      1.00      1.00      6000
       DDoS-RSTFINFlood       1.00      1.00      1.00      6000
         DDoS-SYN_Flood       1.00      1.00      1.00      6000
         DDoS-SlowLoris       0.97      1.00      0.98      4672
DDoS-SynonymousIP_Flood       1.00      1.00      1.00      6000
         DDoS-TCP_Flood       1.00 

In [15]:
import os
import pickle
import numpy as np

# 🔧 Change this path if needed
SAVE_DIR = "/content/drive/MyDrive/MajorProject/AugmentedModels/RandomForest"

os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save model
with open(os.path.join(SAVE_DIR, "rf_aug_model.pkl"), "wb") as f:
    pickle.dump(model, f)

# 2. Save predictions
np.save(os.path.join(SAVE_DIR, "y_pred.npy"), y_pred)
np.save(os.path.join(SAVE_DIR, "y_pred_probs.npy"), y_pred_probs)

# 3. Save label encoder
with open(os.path.join(SAVE_DIR, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)

# 4. Save test data (for XAI)
X_test.to_csv(os.path.join(SAVE_DIR, "X_test.csv"), index=False)
y_test.to_csv(os.path.join(SAVE_DIR, "y_test.csv"), index=False)

# 5. Save feature names (optional but useful)
with open(os.path.join(SAVE_DIR, "feature_names.pkl"), "wb") as f:
    pickle.dump(X_train.columns.tolist(), f)

print(f"✅ All artifacts saved to: {SAVE_DIR}")

✅ All artifacts saved to: /content/drive/MyDrive/MajorProject/AugmentedModels/RandomForest
